In [1]:
import pandas as pd
df=pd.read_csv('/workspaces/BlizzardX/Data/cleaned_data.csv')

In [3]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [4]:
from src.Model.feature_engineering import FeatureEngineering
fe=FeatureEngineering(df)

In [5]:
df=fe.apply_all_features()

In [8]:
df.columns

Index(['DATE', 'Station_ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME',
       'Season', 'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD', 'Station_Location',
       'Station_Lat_Long_Interaction', 'Day_of_Week', 'Day_of_Year',
       'Temp_Diff', 'Seasonal_TMIN_Deviation', 'Rolling_Mean_TMIN_7',
       'Rolling_10thPercentile_TMIN_7', 'TMIN_Rolling_30_Diff', 'EWMA_TMIN_7',
       'Previous_Season_TMIN', 'Seasonal_TMIN_Anomaly', 'Rolling_Max_TMIN_30',
       'Rolling_Min_TMIN_30', 'EWMA_TMIN_30', 'TMIN_Lag1', 'SnowyDay',
       'SnowyDaysCount_7', 'Cumulative_SnowDepth_7',
       'Cumulative_Snowfall_Lag7', 'SNWD_Lag1', 'SNWD_Lag2',
       'Rolling_Sum_SNWD_7', 'SNWD_TMIN_Interaction', 'Snowfall_Intensity',
       'SNWD_Snowfall_Diff', 'PRCP_Lag1', 'PRCP_Lag2',
       'Cumulative_Precipitation_7', 'Rolling_Sum_PRCP_14',
       'TMAX_PRCP_Interaction', 'Days_Since_Last_Precip',
       'Rolling_Mean_TMIN_30', 'TMIN_SNOW_Interaction'],
      dtype='object')

In [ ]:
# Feature Engineering for Weather Data
# 1. Station-Specific Features (Spatial Information)
df['Station_Location'] = df['LATITUDE'].astype(str) + '_' + df['LONGITUDE'].astype(str)
df['Station_Lat_Long_Interaction'] = df['LATITUDE'] * df['LONGITUDE']
df.rename(columns={'ID': 'Station_ID'}, inplace=True)

# 2. Temporal Features
df['Day_of_Week'] = df['DATE'].dt.dayofweek  # 0=Monday, 6=Sunday
df['Day_of_Year'] = df['DATE'].dt.dayofyear  # 1=Jan 1st, 365=Dec 31st
season_mapping = {1: 'Winter', 2: 'Spring', 3: 'Summer', 4: 'Fall'}
df['Season'] = df['Season'].map(season_mapping)

# 3. Temperature Features
df['Temp_Diff'] = df['TMAX'] - df['TMIN']  # Difference between TMAX and TMIN
df['Seasonal_TMIN_Deviation'] = df['TMIN'] - df.groupby('Season')['TMIN'].transform('mean')  # Deviation of TMIN from season mean
df['Rolling_Mean_TMIN_7'] = df['TMIN'].rolling(window=7).mean()  # 7-day rolling mean of TMIN
df['Rolling_10thPercentile_TMIN_7'] = df['TMIN'].rolling(window=7).apply(lambda x: np.percentile(x, 10), raw=True)  # 7-day rolling 10th percentile of TMIN
df['TMIN_Rolling_30_Diff'] = df['TMIN'] - df['Rolling_Mean_TMIN_7']  # Difference between TMIN and 7-day rolling mean
df['EWMA_TMIN_7'] = df['TMIN'].ewm(span=7, adjust=False).mean()  # Exponentially Weighted Moving Average of TMIN (7 days)
df['Previous_Season_TMIN'] = df.groupby('Season')['TMIN'].shift(1)  # Previous day's TMIN in the same season
df['Seasonal_TMIN_Anomaly'] = df['TMIN'] - df.groupby([df['DATE'].dt.month, df['DATE'].dt.day])['TMIN'].transform('mean')  # Seasonal anomaly in TMIN
df['Rolling_Max_TMIN_30'] = df['TMIN'].rolling(window=30).max()  # 30-day rolling max of TMIN
df['Rolling_Min_TMIN_30'] = df['TMIN'].rolling(window=30).min()  # 30-day rolling min of TMIN
df['EWMA_TMIN_30'] = df['TMIN'].ewm(span=30, adjust=False).mean()  # Exponentially Weighted Moving Average of TMIN (30 days)
df['TMIN_Lag1'] = df['TMIN'].shift(1)  # 1-day lag feature for TMIN

# 4. Snow Features
df['SnowyDay'] = (df['SNOW'] > 0).astype(int)  # Flag indicating if snow was observed
df['SnowyDaysCount_7'] = df['SNOW'].rolling(window=7).apply(lambda x: (x > 0).sum(), raw=True)  # Count of snowy days in the last 7 days
df['Cumulative_SnowDepth_7'] = df['SNWD'].rolling(window=7).sum()  # Cumulative snow depth over the last 7 days
df['Cumulative_Snowfall_Lag7'] = df['SNOW'].shift(7).rolling(window=7).sum()  # Cumulative snowfall in the last 7 days
df['SNWD_Lag1'] = df['SNWD'].shift(1)  # 1-day lag feature for snow depth
df['SNWD_Lag2'] = df['SNWD'].shift(2)  # 2-day lag feature for snow depth
df['Rolling_Sum_SNWD_7'] = df['SNWD'].rolling(window=7).sum()  # 7-day rolling sum of snow depth
df['SNWD_TMIN_Interaction'] = df['SNWD'] * df['TMIN']  # Interaction term between snow depth and TMIN
df['Snowfall_Intensity'] = df['SNOW'] / (df['SNWD'] + 1)  # Snowfall intensity (to avoid division by zero)
df['SNWD_Snowfall_Diff'] = df['SNWD'] - df['Snowfall_Intensity']  # Difference between snow depth and snowfall intensity
  # Snowfall intensity (to avoid division by zero)

# 5. Precipitation Features
df['PRCP_Lag1'] = df['PRCP'].shift(1)  # 1-day lag feature for precipitation
df['PRCP_Lag2'] = df['PRCP'].shift(2)  # 2-day lag feature for precipitation
df['Cumulative_Precipitation_7'] = df['PRCP'].rolling(window=7).sum()  # Cumulative precipitation over the last 7 days
df['Rolling_Sum_PRCP_14'] = df['PRCP'].rolling(window=14).sum()  # 14-day rolling sum of precipitation
df['TMAX_PRCP_Interaction'] = df['TMAX'] * df['PRCP']  # Interaction term between maximum temperature and precipitation
df['Days_Since_Last_Precip'] = (df['DATE'] - df['DATE'][df['PRCP'] > 0].shift(1)).dt.days  # Days since last precipitation event

# 6. Additional Time-based and Interaction Features
df['Rolling_Mean_TMIN_30'] = df['TMIN'].rolling(window=30).mean()  # 30-day rolling mean of TMIN
df['TMIN_SNOW_Interaction'] = df['TMIN'] * df['SNOW']  # Interaction term between TMIN and SNOW
df['Seasonal_TMIN_Anomaly'] = df['TMIN'] - df.groupby('Season')['TMIN'].transform('mean')  # Seasonal anomaly of TMIN
df['Day_of_Week'] = df['DATE'].dt.dayofweek  # Day of the week (0=Monday, 6=Sunday)
df['Day_of_Year'] = df['DATE'].dt.dayofyear  # Day of the year (1=Jan 1st, 365=Dec 31st)
df['Station_Location'] = df['LATITUDE'].astype(str) + '_' + df['LONGITUDE'].astype(str)  # Combine LATITUDE and LONGITUDE
df['Station_Lat_Long_Interaction'] = df['LATITUDE'] * df['LONGITUDE']  # Interaction term between LATITUDE and LONGITUDE


In [2]:
# 1. TMIN Percentile (Weekly or Seasonal)
df['Week'] = df['DATE'].dt.isocalendar().week  # Create a week column
df['TMIN_10thPercentile_Week'] = df.groupby('Week')['TMIN'].transform(lambda x: x.quantile(0.1))

# 2. Temperature Difference (TMAX - TMIN)
df['Temp_Diff'] = df['TMAX'] - df['TMIN']

# 3. Snowy Day Flag (if SNOW > 0)
df['SnowyDay'] = (df['SNOW'] > 0).astype(int)

# 4. Snow Depth Lag Features (Lag-1, Lag-2)
df['SNWD_Lag1'] = df['SNWD'].shift(1)
df['SNWD_Lag2'] = df['SNWD'].shift(2)

# 5. Snowy Days Count (Last 7 Days)
df['SnowyDaysCount_7'] = df['SNOW'].rolling(window=7).apply(lambda x: (x > 0).sum(), raw=True)

# 6. Cumulative Snow Depth (Last 7 Days)
df['Cumulative_SnowDepth_7'] = df['SNWD'].rolling(window=7).sum()

# 7. Precipitation Lag Features (Lag-1, Lag-2)
df['PRCP_Lag1'] = df['PRCP'].shift(1)
df['PRCP_Lag2'] = df['PRCP'].shift(2)

# 8. Cumulative Precipitation (Last 7 Days)
df['Cumulative_Precipitation_7'] = df['PRCP'].rolling(window=7).sum()

# 9. Rolling Mean of TMIN (3, 5, 7 Days)
df['Rolling_Mean_TMIN_7'] = df['TMIN'].rolling(window=7).mean()

# 10. Rolling Percentile of TMIN (10th Percentile, 7 Days)
df['Rolling_10thPercentile_TMIN_7'] = df['TMIN'].rolling(window=7).apply(lambda x: np.percentile(x, 10), raw=True)

# 11. Day of the Week
df['Day_of_Week'] = df['DATE'].dt.dayofweek

# 12. Day of Year (Julian Date)
df['Day_of_Year'] = df['DATE'].dt.dayofyear

# 13. Interaction between Snow Depth and TMIN
df['SNWD_TMIN_Interaction'] = df['SNWD'] * df['TMIN']

# 14. Snowfall Intensity (SNOW / SNWD)
df['Snowfall_Intensity'] = df['SNOW'] / (df['SNWD'] + 1)  # Avoid division by zero



In [3]:
import pandas as pd
import numpy as np

# Assuming df is your DataFrame containing the data

# 1. Seasonal Temperature Deviation
df['Seasonal_TMIN_Deviation'] = df['TMIN'] - df.groupby('Season')['TMIN'].transform('mean')

# 2. Rolling Mean of TMIN (30 days)
df['Rolling_Mean_TMIN_30'] = df['TMIN'].rolling(window=30).mean()

# 3. Difference between TMIN and Rolling Mean (30 days)
df['TMIN_Rolling_30_Diff'] = df['TMIN'] - df['Rolling_Mean_TMIN_30']

# 4. Exponentially Weighted Moving Average of TMIN (7 days)
df['EWMA_TMIN_7'] = df['TMIN'].ewm(span=7, adjust=False).mean()

# 5. Previous Season TMIN (Previous day in the same season)
df['Previous_Season_TMIN'] = df.groupby('Season')['TMIN'].shift(1)

# 6. Rolling Sum of Precipitation (14 days)
df['Rolling_Sum_PRCP_14'] = df['PRCP'].rolling(window=14).sum()

# 7. Cumulative Snowfall (Lag-7)
df['Cumulative_Snowfall_Lag7'] = df['SNOW'].shift(7).rolling(window=7).sum()

# 8. Temperature and Snowfall Interaction
df['TMIN_SNOW_Interaction'] = df['TMIN'] * df['SNOW']
df['TMAX_PRCP_Interaction'] = df['TMAX'] * df['PRCP']

# 9. Temperature Anomaly from Historical Data (Seasonal anomaly)
df['Seasonal_TMIN_Anomaly'] = df['TMIN'] - df.groupby([df['DATE'].dt.month, df['DATE'].dt.day])['TMIN'].transform('mean')

# 10. Time Since Last Precipitation
df['Days_Since_Last_Precip'] = (df['DATE'] - df['DATE'][df['PRCP'] > 0].shift(1)).dt.days

# 11. Wind and Temperature Interaction (if wind data is available)
# Assuming wind speed column is 'WSPD'
# df['Wind_Temperature_Interaction'] = df['WSPD'] * df['TMIN']  # Uncomment if wind speed data is available

# 12. Rolling Max and Min Temperatures (30 days)
df['Rolling_Max_TMIN_30'] = df['TMIN'].rolling(window=30).max()
df['Rolling_Min_TMIN_30'] = df['TMIN'].rolling(window=30).min()

# 13. Exponentially Weighted Moving Average of TMIN (30 days)
df['EWMA_TMIN_30'] = df['TMIN'].ewm(span=30, adjust=False).mean()

# 14. Snow Depth and Snowfall Interaction
df['SNWD_Snowfall_Diff'] = df['SNWD'] - df['Snowfall_Intensity']

# Example feature engineering for lag features if not already done
# Lag features for 'TMIN', 'TMAX', etc.
df['TMIN_Lag1'] = df['TMIN'].shift(1)
df['TMAX_Lag1'] = df['TMAX'].shift(1)

# Lag features for 'PRCP' and 'SNOW' (if not already created)
df['PRCP_Lag1'] = df['PRCP'].shift(1)
df['SNOW_Lag1'] = df['SNOW'].shift(1)

# Lag features for 'SNWD'
df['SNWD_Lag1'] = df['SNWD'].shift(1)
df['SNWD_Lag2'] = df['SNWD'].shift(2)

# Rolling statistics for SNWD or any other feature (7 days, for example)
df['Rolling_Sum_SNWD_7'] = df['SNWD'].rolling(window=7).sum()

# 15. Cumulative Snow Depth (Lag 7 days for Snow Depth)
df['Cumulative_SnowDepth_7'] = df['SNWD'].rolling(window=7).sum()

# 16. Cumulative Precipitation (Lag 7 days for Precipitation)
df['Cumulative_Precipitation_7'] = df['PRCP'].rolling(window=7).sum()

# 17. Rolling Mean of TMIN (7 days for shorter window)
df['Rolling_Mean_TMIN_7'] = df['TMIN'].rolling(window=7).mean()

# 18. Rolling 10th Percentile of TMIN (7 days)
df['Rolling_10thPercentile_TMIN_7'] = df['TMIN'].rolling(window=7).apply(lambda x: np.percentile(x, 10), raw=True)

# 19. Day of the Week
df['Day_of_Week'] = df['DATE'].dt.dayofweek

# 20. Day of the Year (Julian Date)
df['Day_of_Year'] = df['DATE'].dt.dayofyear

# 21. Interaction between Snow Depth and TMIN
df['SNWD_TMIN_Interaction'] = df['SNWD'] * df['TMIN']

# 22. Snowfall Intensity (Snow / Snow Depth)
df['Snowfall_Intensity'] = df['SNOW'] / (df['SNWD'] + 1)  # Avoid division by zero


In [4]:
# 1. Forward-fill for lag features and rolling window features
df = df.ffill()

# 2. Backward-fill for features that may have missing values at the start
df = df.bfill()

# 3. Fill cumulative features with 0
df['Cumulative_SnowDepth_7'] = df['Cumulative_SnowDepth_7'].fillna(0)
df['Cumulative_Precipitation_7'] = df['Cumulative_Precipitation_7'].fillna(0)

# 4. Fill missing values for SNOW with the median value
df['SNOW'] = df['SNOW'].fillna(df['SNOW'].median())

# 5. Fill missing values for continuous features like 'SNWD_TMIN_Interaction' with the median
df['SNWD_TMIN_Interaction'] = df['SNWD_TMIN_Interaction'].fillna(df['SNWD_TMIN_Interaction'].median())
df['Snowfall_Intensity'] = df['Snowfall_Intensity'].fillna(df['Snowfall_Intensity'].median())

# 6. Fill missing values for categorical features like 'Season' with the mode (most frequent value)
df['Season'] = df['Season'].fillna(df['Season'].mode()[0])

# 7. For lag features like 'SNWD_Lag1', 'SNWD_Lag2', 'PRCP_Lag1', and 'PRCP_Lag2', use forward-fill
df['SNWD_Lag1'] = df['SNWD_Lag1'].ffill()
df['SNWD_Lag2'] = df['SNWD_Lag2'].ffill()
df['PRCP_Lag1'] = df['PRCP_Lag1'].ffill()
df['PRCP_Lag2'] = df['PRCP_Lag2'].ffill()


In [5]:
df.isnull().sum()  # Check for any null values

DATE                             0
ID                               0
LATITUDE                         0
LONGITUDE                        0
ELEVATION                        0
NAME                             0
Season                           0
TMIN                             0
TMAX                             0
PRCP                             0
SNOW                             0
SNWD                             0
Week                             0
TMIN_10thPercentile_Week         0
Temp_Diff                        0
SnowyDay                         0
SNWD_Lag1                        0
SNWD_Lag2                        0
SnowyDaysCount_7                 0
Cumulative_SnowDepth_7           0
PRCP_Lag1                        0
PRCP_Lag2                        0
Cumulative_Precipitation_7       0
Rolling_Mean_TMIN_7              0
Rolling_10thPercentile_TMIN_7    0
Day_of_Week                      0
Day_of_Year                      0
SNWD_TMIN_Interaction            0
Snowfall_Intensity  

In [6]:
# List of columns to exclude from rounding
exclude_columns = ['DATE', 'ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season']

# Get all columns in the DataFrame
all_columns = df.columns

# Select numerical columns excluding the ones listed above
columns_to_round = [col for col in all_columns if col not in exclude_columns]

# Round the selected columns to 2 decimal places
df[columns_to_round] = df[columns_to_round].round(2)


In [25]:
df.to_csv('/workspaces/BlizzardX/Data/Final_Processed.csv', index=False)  # Save the processed DataFrame

In [29]:
df.columns  # Display the columns in the DataFrame

Index(['DATE', 'ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season',
       'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD', 'Week',
       'TMIN_10thPercentile_Week', 'Temp_Diff', 'SnowyDay', 'SNWD_Lag1',
       'SNWD_Lag2', 'SnowyDaysCount_7', 'Cumulative_SnowDepth_7', 'PRCP_Lag1',
       'PRCP_Lag2', 'Cumulative_Precipitation_7', 'Rolling_Mean_TMIN_7',
       'Rolling_10thPercentile_TMIN_7', 'Day_of_Week', 'Day_of_Year',
       'SNWD_TMIN_Interaction', 'Snowfall_Intensity',
       'Seasonal_TMIN_Deviation', 'Rolling_Mean_TMIN_30',
       'TMIN_Rolling_30_Diff', 'EWMA_TMIN_7', 'Previous_Season_TMIN',
       'Rolling_Sum_PRCP_14', 'Cumulative_Snowfall_Lag7',
       'TMIN_SNOW_Interaction', 'TMAX_PRCP_Interaction',
       'Seasonal_TMIN_Anomaly', 'Days_Since_Last_Precip',
       'Rolling_Max_TMIN_30', 'Rolling_Min_TMIN_30', 'EWMA_TMIN_30',
       'SNWD_Snowfall_Diff', 'TMIN_Lag1', 'TMAX_Lag1', 'SNOW_Lag1',
       'Rolling_Sum_SNWD_7'],
      dtype='object')

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data into train and test (e.g., 80% train, 20% test)
train_size = int(len(df) * 0.8)
train, test = df[:train_size], df[train_size:]

# Scaling the features (for traditional models)
scaler = StandardScaler()
X_train = train.drop(columns=['DATE', 'ID','NAME', 'Season', 'TMIN'])  # Assuming TMIN is the target variable
X_test = test.drop(columns=['DATE', 'ID','NAME',  'Season', 'TMIN'])

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Target variable (TMIN)
y_train = train['TMIN']
y_test = test['TMIN']


In [34]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# Ensure DATE column is in datetime format
df['DATE'] = pd.to_datetime(df['DATE'])

# Set 'DATE' as the index
df.set_index('DATE', inplace=True)

# Optionally, set frequency (e.g., daily)
df = df.asfreq('D')  # Change 'D' to the appropriate frequency ('W' for weekly, etc.)

# Select your target variable (e.g., 'TMIN')
y_train = df['TMIN'][:len(df)//2]  # First half for training
y_test = df['TMIN'][len(df)//2:]  # Second half for testing

# ARIMA model (adjust the order as needed)
arima_model = ARIMA(y_train, order=(5, 1, 0))
arima_model_fit = arima_model.fit()

# Forecasting on test data
arima_forecast = arima_model_fit.forecast(steps=len(y_test))

# Now arima_forecast is a pandas Series with forecasts


In [35]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Define and fit SARIMA model
sarima_model = SARIMAX(y_train, order=(5, 1, 0), seasonal_order=(1, 1, 0, 7))  # You can adjust these orders
sarima_model_fit = sarima_model.fit()

# Forecasting on test data
sarima_forecast = sarima_model_fit.forecast(steps=len(y_test))

# Print and visualize forecast
print(sarima_forecast)


1986-08-02      9.520762
1986-08-03     10.790201
1986-08-04      8.820720
1986-08-05      7.484025
1986-08-06      6.835082
                 ...    
2025-02-27    287.427188
2025-02-28    288.272964
2025-03-01    288.909985
2025-03-02    289.746125
2025-03-03    288.442222
Freq: D, Name: predicted_mean, Length: 14094, dtype: float64
